In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# Configuration
# Using existing Unity Catalog connection
GDRIVE_CONNECTION = "gdrive"
GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1rIiZBxanLos2u_SvU6YJvr3aci782CAh"

@dp.table(
    name="emp_item",
    comment="Bronze layer: Raw employee JSON data ingested from Google Drive folder"
)
def emp_item():
    """
    Incrementally ingest JSON employee data from Google Drive.
    Uses Auto Loader with cloudFiles format for incremental processing.
    Handles multiLine JSON format for nested structures.
    """
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("databricks.connection", GDRIVE_CONNECTION)
        .option("multiLine", "true")
        .option("inferColumnTypes", True)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("pathGlobFilter", "*.json")
        .load(GDRIVE_FOLDER_URL)
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
    )
